# Part 1 – GLM (task 1.3)
Needs the outputs of `1_preprocessing.ipynb` (`bold_moco.par` and `bold_moco_smoothed-6mm.nii.gz`).
Before pushing: **Edit → Clear Outputs of All Cells**, then save.

In [ ]:
import os, sys
import os.path as op
sys.path.append('.')
from paths import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib

from nilearn import image, datasets
from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix
from nilearn.glm import threshold_stats_img
from nilearn.plotting import plot_design_matrix, plot_contrast_matrix, plot_stat_map

AXIAL_CUTS = (-20, -10, 0, 10, 20, 30, 40, 50)   # same axial slices for every map

In [ ]:
fmri_img = nib.load(FUNC_PATH)
events = pd.read_csv(EVENTS_PATH)

n_scans = fmri_img.shape[3]
tr = float(fmri_img.header.get_zooms()[3])
frame_times = np.arange(n_scans) * tr
print("Scans:", n_scans, "| TR:", tr, "s | run length:", n_scans * tr, "s")

print("\nAll event types in the csv:")
print(events["trial_type"].value_counts())

## 3a Design matrix
Task regressors: story and math blocks (HRF-convolved, SPM HRF). Nuisance: 6 motion parameters + cosine drifts (high-pass 0.01 Hz) + constant.

In [ ]:
events_glm = events[events["trial_type"].isin(["story", "math"])].copy()

motion_params = pd.read_csv(MOTION_PAR, sep=r"\s+", header=None,
                            names=["rot_x", "rot_y", "rot_z", "trans_x", "trans_y", "trans_z"])
assert len(motion_params) == n_scans

design_matrix = make_first_level_design_matrix(
    frame_times=frame_times, events=events_glm,
    hrf_model="spm", drift_model="cosine", high_pass=0.01,
    add_regs=motion_params.to_numpy(), add_reg_names=motion_params.columns.tolist())

print("Regressors:", design_matrix.columns.tolist())
ax = plot_design_matrix(design_matrix)
ax.get_figure().savefig(op.join(FIG_DIR, "3a_design_matrix.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(frame_times, design_matrix["story"], label="Story")
plt.plot(frame_times, design_matrix["math"], label="Math")
plt.xlabel("Time (s)"); plt.ylabel("Regressor amplitude")
plt.title("HRF-convolved task regressors"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(op.join(FIG_DIR, "3a_task_regressors.png"), dpi=200)
plt.show()

## 3b Correlation between regressors

In [ ]:
corr_matrix = design_matrix.drop(columns=["constant"]).corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix))); ax.set_xticklabels(corr_matrix.columns, rotation=90)
ax.set_yticks(range(len(corr_matrix))); ax.set_yticklabels(corr_matrix.index)
plt.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Correlation between GLM regressors")
plt.tight_layout()
plt.savefig(op.join(FIG_DIR, "3b_correlation_matrix.png"), dpi=200, bbox_inches="tight")
plt.show()

print("story vs math:", round(corr_matrix.loc["story", "math"], 3))
display(corr_matrix.loc[["story", "math"]].round(3))

## 3c GLM (first level, single subject, AR(1) noise) and maps of each task regressor
Maps are FDR-corrected (q < 0.05) and **two-sided**: red = positive, blue = negative effect.
Story and math blocks fill almost the whole run (r ≈ −0.94), so the implicit baseline is very short:
the single-regressor maps are hard to interpret and mostly negative for story (this is why the earlier one-sided FDR gave `inf`).

In [ ]:
smoothed_img = nib.load(SMOOTH_PATH + ".nii.gz")
assert smoothed_img.shape[3] == len(design_matrix)

fmri_glm = FirstLevelModel(noise_model="ar1", standardize=False)   # TR is already in the design matrix
fmri_glm = fmri_glm.fit(smoothed_img, design_matrices=design_matrix)
mask_img = fmri_glm.masker_.mask_img_
print("GLM fitted")

In [ ]:
def fdr_map(zmap, title, fname, two_sided=True):
    # FDR-threshold a z-map, plot it on the T1 (axial slices) and save the figure
    thr_map, thr = threshold_stats_img(zmap, mask_img=mask_img, alpha=0.05,
                                       height_control="fdr", two_sided=two_sided)
    print(f"{title}: FDR threshold z = {thr:.2f}")
    if not np.isfinite(thr):
        print("  -> no voxel survives FDR")
        return thr_map, thr
    disp = plot_stat_map(thr_map, bg_img=ANAT_PATH, threshold=thr, display_mode="z",
                         cut_coords=AXIAL_CUTS, title=f"{title} (FDR q < 0.05)", colorbar=True)
    disp.savefig(op.join(FIG_DIR, fname), dpi=200)
    plt.show()
    return thr_map, thr

story_zmap = fmri_glm.compute_contrast("story", output_type="z_score")
math_zmap = fmri_glm.compute_contrast("math", output_type="z_score")
fdr_map(story_zmap, "Story", "3c_story.png")
fdr_map(math_zmap, "Math", "3c_math.png")

## 3d Story vs math contrast
Contrast vector c = [+1 for story, −1 for math, 0 for all other regressors].
Two-sided map: **red = story > math**, **blue = math > story**.

In [ ]:
c_story_vs_math = np.zeros(design_matrix.shape[1])
c_story_vs_math[design_matrix.columns.get_loc("story")] = 1
c_story_vs_math[design_matrix.columns.get_loc("math")] = -1
print(pd.Series(c_story_vs_math, index=design_matrix.columns).to_string())

ax = plot_contrast_matrix(c_story_vs_math, design_matrix=design_matrix)
ax.get_figure().savefig(op.join(FIG_DIR, "3d_contrast_vector.png"), dpi=200, bbox_inches="tight")
plt.show()

story_vs_math_zmap = fmri_glm.compute_contrast(c_story_vs_math, output_type="z_score")
sm_thr_map, sm_thr = fdr_map(story_vs_math_zmap, "Story > Math (red) / Math > Story (blue)", "3d_story_vs_math.png")

In [ ]:
# Save maps to look at them in FSLeyes (results/ is git-ignored)
nib.save(story_zmap, op.join(RESULTS_DIR, "story_zmap.nii.gz"))
nib.save(math_zmap, op.join(RESULTS_DIR, "math_zmap.nii.gz"))
nib.save(story_vs_math_zmap, op.join(RESULTS_DIR, "story_vs_math_zmap.nii.gz"))
nib.save(sm_thr_map, op.join(RESULTS_DIR, "story_vs_math_zmap_fdr.nii.gz"))

### Optional: look at the maps in FSLeyes
FSLeyes needs FSL loaded **and** `%gui wx` before `FSLeyesServer()` (that was missing before, which is why it didn't work).

In [ ]:
import module as lmod
from utils import loadFSL, FSLeyesServer
await lmod.load('fsl/6.0.7.18')
loadFSL()

In [ ]:
%gui wx

In [ ]:
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(ANAT_PATH)
fsleyesDisplay.load(op.join(RESULTS_DIR, "story_vs_math_zmap_fdr.nii.gz"))

## 3e Functional networks (Yeo 7 networks, via the Schaefer 2018 parcellation)
*Proposal, not tested yet.* For each direction of the contrast, the share of significant voxels falling in each Yeo network.
Expected pattern: story > math mostly in the **default mode** network, math > story mostly in **fronto-parietal / dorsal attention**.

In [ ]:
# Schaefer 2018 parcellation (400 cortical parcels), each parcel labelled with its Yeo-7 network.
# Downloaded from GitHub (more reliable than the original Yeo atlas, which uses an FTP server).
NETWORKS = {"Vis": "Visual", "SomMot": "Somatomotor", "DorsAttn": "Dorsal attention",
            "SalVentAttn": "Ventral attention", "Limbic": "Limbic", "Cont": "Frontoparietal",
            "Default": "Default mode"}

schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=7, resolution_mm=2)
labels = [l.decode() if isinstance(l, bytes) else str(l) for l in schaefer.labels]
if labels[0].lower() != "background":          # older nilearn: no background entry
    labels = ["Background"] + labels

atlas = image.resample_to_img(schaefer.maps, story_vs_math_zmap, interpolation="nearest")
parcels = np.round(atlas.get_fdata()).astype(int)
network_map = np.full(parcels.shape, "Outside cortex atlas", dtype=object)
for k in range(1, len(labels)):
    network_map[parcels == k] = NETWORKS[labels[k].split("_")[2]]

z = sm_thr_map.get_fdata()
rows = {}
for name, sig in [("Story > Math", z > 0), ("Math > Story", z < 0)]:
    counts = pd.Series(network_map[sig]).value_counts()
    rows[name] = 100 * counts / sig.sum()
order = list(NETWORKS.values()) + ["Outside cortex atlas"]
overlap = pd.DataFrame(rows).reindex(order).fillna(0).round(1)
display(overlap)   # % of the significant voxels of each contrast in each network

overlap.T.plot(kind="barh", stacked=True, figsize=(10, 3), colormap="tab10")
plt.xlabel("% of significant voxels"); plt.legend(bbox_to_anchor=(1, 1)); plt.tight_layout()
plt.savefig(op.join(FIG_DIR, "3e_network_overlap.png"), dpi=200, bbox_inches="tight")
plt.show()

## 3f [Hard] Semantic language regions (Binder et al.)
TODO: check angular gyrus, middle/inferior temporal gyrus, inferior frontal gyrus, posterior cingulate/precuneus in the story > math map, and discuss why some could be missing.